# Advanced Numerical: Verifying Classical Hopfield Convergence and Hebbian Recall Limits

This notebook isolates the numerical checks that support the Week 6 classical Hopfield derivations. We verify monotonic energy descent during asynchronous retrieval and then quantify how recall degrades as load increases.


> __Learning Objectives:__
> 
> By the end of this notebook, you should be able to:
>
> * __Verify trajectory-level descent__: Track energy along one asynchronous retrieval trajectory and confirm that accepted flips never increase energy. Connect the measured $\Delta E$ values back to the derivation formula from the advanced derivation notebook.
>
> * __Measure load-dependent recall__: Run repeated retrieval trials across multiple loads $\alpha=K/N$ and estimate success probability empirically. Use the observed trend to interpret how crosstalk limits reliable memory recovery.


Let's get started!

___


## Numerical Verification (Julia)


Now we connect the convergence and Hebbian-crosstalk derivations to computation and test those predictions in simulation. For the full line-by-line proofs, see [Advanced: Why Classical Hopfield Networks Converge and Why Hebbian Learning Works](CHEME-5820-L6a-Advanced-Classical-HopfieldConvergence-HebbianLearning-Spring-2026.ipynb).


> __What we test:__
> 
> * __Trajectory-level check__: Along one asynchronous retrieval trajectory, accepted flips should never increase energy. We verify this by computing $\Delta E$ over the observed sequence of states.
>
> * __Capacity-level check__: As load $\alpha=K/N$ increases, recall success should decrease because crosstalk increases. We estimate this trend with repeated randomized trials at fixed corruption level.


Before running code, define the experiment parameters explicitly so each reported result has clear meaning.


> __Parameter definitions for the experiments:__ Here $N$ is the number of neurons, $K$ is the number of stored patterns, and the load is $\alpha=K/N$. A noisy cue is an initial state created by flipping a fraction of bits in one stored pattern, `trials` counts repeated randomized runs, and recall success is the fraction of trials in which the retrieved state exactly matches the target stored pattern.


First, define reusable helper functions for Hebbian storage, asynchronous retrieval, energy evaluation, and Hamming distance (the number of bit positions where two $\{-1,+1\}$ states differ).


___


In [1]:
using Random
using LinearAlgebra
using Printf

function hebbian_weights(patterns::Matrix{Int})
    _, N = size(patterns)
    W = (patterns' * patterns) ./ N
    W[diagind(W)] .= 0.0
    return Matrix{Float64}(W)
end

function energy(state::Vector{Int}, W::Matrix{Float64}, bias::Vector{Float64}=zeros(Float64, length(state)))
    return -0.5 * dot(state, W * state) - dot(bias, state)
end

function corrupt_pattern(pattern::Vector{Int}, flip_fraction::Float64, rng::AbstractRNG)
    s = copy(pattern)
    n_flip = round(Int, flip_fraction * length(s))
    if n_flip > 0
        idx = randperm(rng, length(s))[1:n_flip]
        s[idx] .*= -1
    end
    return s
end

hamming_distance(a::Vector{Int}, b::Vector{Int}) = count(a .!= b)

function async_retrieve(initial_state::Vector{Int}, W::Matrix{Float64};
    bias::Vector{Float64}=zeros(Float64, length(initial_state)),
    max_sweeps::Int=80,
    rng::AbstractRNG=Random.default_rng())

    s = copy(initial_state)
    N = length(s)
    energies = Float64[energy(s, W, bias)]

    for _ in 1:max_sweeps
        flipped = false
        for i in randperm(rng, N)
            h_i = dot(@view(W[i, :]), s) + bias[i]
            proposed = h_i == 0 ? s[i] : (h_i > 0 ? 1 : -1)
            if proposed != s[i]
                s[i] = proposed
                flipped = true
                push!(energies, energy(s, W, bias))
            end
        end
        !flipped && break
    end

    return s, energies
end


async_retrieve (generic function with 1 method)

> __Experiment 1: Monotonic energy descent__
> 
> Start from a noisy cue, run asynchronous retrieval, and track $E(\mathbf{s})$ after each accepted flip. The convergence derivation predicts that $\Delta E\le 0$ at each asynchronous step, so here we test whether the largest observed $\Delta E$ along the trajectory is non-positive.

In [2]:
rng = MersenneTwister(5820)

N = 120
K = 10
patterns = rand(rng, (-1, 1), K, N)
W = hebbian_weights(patterns)

target = vec(patterns[1, :])
noisy = corrupt_pattern(target, 0.20, rng)
retrieved, energies = async_retrieve(noisy, W; max_sweeps=80, rng=rng)

energy_diffs = diff(energies)
largest_delta_E = isempty(energy_diffs) ? 0.0 : maximum(energy_diffs)
is_monotone = all(energy_diffs .<= 1e-12)

println("Monotonic energy check (single retrieval run)")
@printf("N = %d, K = %d, alpha = %.3f\n", N, K, K / N)
println("Energy monotone non-increasing: ", is_monotone)
@printf("Largest observed delta E over accepted flips: %.3e\n", largest_delta_E)
println("Initial Hamming distance to target: ", hamming_distance(noisy, target))
println("Final Hamming distance to target:   ", hamming_distance(retrieved, target))
println("Number of accepted flips: ", length(energies) - 1)


Monotonic energy check (single retrieval run)
N = 120, K = 10, alpha = 0.083
Energy monotone non-increasing: true
Largest observed delta E over accepted flips: -3.667e-01
Initial Hamming distance to target: 24
Final Hamming distance to target:   0
Number of accepted flips: 24


> __Experiment 2: Recall vs. load__
> 
> Sweep $\alpha=K/N$ at fixed corruption level and estimate recall success over repeated trials. Experiment 2 directly tests the derived signal-crosstalk mechanism by measuring recall as load $\alpha=K/N$ increases at fixed cue corruption.

In [3]:
function recall_success_rate(N::Int, alpha_values::Vector{Float64};
    trials::Int=60,
    flip_fraction::Float64=0.10,
    max_sweeps::Int=80,
    seed::Int=5821)

    rng = MersenneTwister(seed)
    rows = Tuple{Float64, Int, Float64}[]

    for alpha in alpha_values
        K = max(1, round(Int, alpha * N))
        success = 0

        for _ in 1:trials
            patterns = rand(rng, (-1, 1), K, N)
            W = hebbian_weights(patterns)
            idx = rand(rng, 1:K)
            target = vec(patterns[idx, :])
            noisy = corrupt_pattern(target, flip_fraction, rng)
            retrieved, _ = async_retrieve(noisy, W; max_sweeps=max_sweeps, rng=rng)
            retrieved == target && (success += 1)
        end

        push!(rows, (alpha, K, success / trials))
    end

    return rows
end

N = 120
alpha_values = [0.03, 0.06, 0.09, 0.12, 0.15, 0.18, 0.21]
results = recall_success_rate(N, alpha_values; trials=50, flip_fraction=0.10, max_sweeps=80, seed=5822)

println("Recall performance vs load alpha = K/N")
println("N = $(N), noise = 10% bit flips, trials per alpha = 50")
println("alpha    K    success_rate")

for (alpha, K, rate) in results
    bar = repeat("#", round(Int, 30 * rate))
    @printf("%5.2f  %3d    %6.3f  %s\n", alpha, K, rate, bar)
end

Recall performance vs load alpha = K/N
N = 120, noise = 10% bit flips, trials per alpha = 50
alpha    K    success_rate
 0.03    4     1.000  ##############################
 0.06    7     1.000  ##############################
 0.09   11     0.980  #############################
 0.12   14     0.960  #############################
 0.15   18     0.620  ###################
 0.18   22     0.240  #######
 0.21   25     0.160  #####


## Summary


The numerical experiments match the derivation-level predictions for classical Hopfield networks. Energy decreases monotonically along accepted asynchronous flips, while retrieval reliability drops as load increases.


> __Key takeaways:__
>
> * __Convergence check__: The trajectory-level experiment verifies the theoretical sign of $\Delta E$ directly from simulation. Every accepted update is non-increasing in energy, which is exactly the Lyapunov-style behavior proved in the derivation notebook.
> * __Capacity check__: The load sweep shows the expected decline in exact recall success as $\alpha=K/N$ grows. This trend is consistent with increasing crosstalk in the Hebbian local-field decomposition.

___